In [2]:
import pandas as pd
import numpy as np
import math
from matplotlib import pyplot as plt

In [3]:
data = pd.read_csv('../datasets/train.csv')

In [4]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
data.dtypes.count()

np.int64(785)

In [6]:
type(data)

pandas.core.frame.DataFrame

In [7]:
data = np.array(data)

In [8]:
data_y = data[:,0]
data_X = data[:,1:]

In [9]:
data_X = data_X.astype(np.float32) / 255.0

In [10]:
data_y.dtype

dtype('int64')

In [11]:
data_X.dtype

dtype('float32')

In [12]:
np.eye(10)[data_y]

array([[0., 1., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(42000, 10))

In [13]:
m, n = data_X.shape

In [14]:
print(data_y)

[1 0 1 ... 7 6 9]


In [15]:
shuffle_i = np.arange(m)

In [16]:
np.random.shuffle(shuffle_i)

In [17]:
data_X = data_X[shuffle_i]
data_y = data_y[shuffle_i]

In [19]:
print(data_X.shape)

(42000, 784)


In [20]:
def init_params():
    w1 = np.random.rand(784,128) - 0.5
    w2 = np.random.rand(128,10) - 0.5
    b1 = np.zeros((1, 128))
    b2 = np.zeros((1, 10))
    return w1,b1,w2,b2

In [21]:
def reLU(a):
    return np.maximum(0,a)
def sigmoid(a):
    return 1 / (1 + np.exp(-a))
def softmax(a):
    exp_a = np.exp(a - np.max(a))
    return exp_a / np.sum(exp_a)

In [22]:
def forward_propagation(data_X,w1,b1,w2,b2):
    z1 = np.dot(data_X,w1) + b1
    a1 = reLU(z1)
    z2 = np.dot(a1,w2) + b2
    a2 = softmax(z2)
    return z1,a1,z2,a2

In [23]:
def compute_loss(y_true, y_pred):
    m = y_true.shape[0]
    eps = 1e-8
    loss = -np.sum(y_true * np.log(y_pred + eps)) / m
    return loss

In [25]:
def relu_deriv(a):
    return (a > 0).astype(float)

In [26]:
def backward_propagation(z1, a1, z2, a2, w1, w2, X, y):
    m = X.shape[0]

    dz2 = a2 - y
    dw2 = np.dot(a1.T, dz2) / m
    db2 = np.mean(dz2, axis=0, keepdims=True)

    dz1 = np.dot(dz2, w2.T) * relu_deriv(z1)
    dw1 = np.dot(X.T, dz1) / m
    db1 = np.mean(dz1, axis=0, keepdims=True)

    return dw1, db1, dw2, db2

In [29]:
def update_params(w1,b1,w2,b2,dw1,db1,dw2,db2,lrate):
    w1 = w1 - lrate * dw1
    b1 = b1 - lrate * db1
    w2 = w2 - lrate * dw2
    b2 = b2 - lrate * db2
    return w1,b1,w2,b2

In [ ]:
def gradient_descent(X,y,lr,iterations):
    w1,b1,w2,b2 = init_params()
    for iter in range(iterations):
        z1,a1,z2,a2 = forward_propagation(X,w1,b1,w2,b2)
        loss = compute_loss(y,a2)
        dw1,db1,dw2,db2 = backward_propagation(z1,a1,z2,a2,w1,w2,X,y)
        w1,b1,w2,b2 = update_params(w1,b1,w2,b2,dw1,db1,dw2,db2,lr)
        if iter % 100 == 0:
            print(f"Epoch {iter} - Loss: {loss}")
